# harness-admission on Colab (T4)
Run top to bottom. Everything lands in Drive (`MyDrive/ha_home`), so after a disconnect just rerun cells 1-3 and the step you were on: finished work is skipped.

In [ ]:
!nvidia-smi   # must show Tesla T4 (Runtime > Change runtime type > T4 GPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ha_home
!git clone https://github.com/Soumya080/harness-admission.git /content/ha 2>/dev/null || (cd /content/ha && git pull)
%cd /content/ha

In [ ]:
!pip install -q vllm datasets pandas pyarrow scipy matplotlib pytest
import os
os.environ['HA_HOME'] = '/content/drive/MyDrive/ha_home'
os.environ['HA_DTYPE'] = 'half'      # T4 has no bf16
os.environ['HA_GPU_UTIL'] = '0.85'

In [ ]:
# CPU checks: must print 'passed' and 'ALL DONE'
!python -m pytest -q tests
!HA_FAKE=1 HA_HOME=$(mktemp -d) bash scripts/run_all.sh 2>&1 | tail -2

In [ ]:
# GPU: base + planted, then STOP CHECK
!python -m data.pools
!python -m proposer.manual
!python -m scripts.run_grid --only base
!python -m scripts.run_grid --only planted
!python -m scripts.check_stop

**Only continue if the cell above printed `CHECK PASSED`.** Otherwise send the output.

In [ ]:
# Proposer (7B AWQ). Restart runtime first if you hit OOM, then rerun cells 2-3 and this one.
from harness.store import list_harnesses, load_harness
n = sum(load_harness(x).tag == 'llm' for x in list_harnesses())
print('llm edits so far:', n)
if n < 30:
    !python -m proposer.propose --n {30 - n}

In [ ]:
# Full grid: the long step (~1-3 h). Rerun this cell after any disconnect.
!python -m scripts.run_grid

In [ ]:
# Quick diagnostics on Colab (CPU steps for the report are better on your laptop)
!python -m scripts.diagnose | tee $HA_HOME/results_diagnose.txt

## Then on your laptop
Download `MyDrive/ha_home`, then:
```bash
export HA_HOME=/path/to/ha_home
python -m scripts.diagnose | tee $HA_HOME/results/diagnose.txt
python -m sims.stage0 | tee $HA_HOME/results/stage0.txt
python -m sims.plot_stage0
python -m scripts.make_report --runs 1000 --ablations
```